# Mass Euchre Game Runner

This notebook provides tools to run large numbers of euchre games for:
- Training data generation
- AI performance evaluation
- Strategy analysis
- Statistical analysis

## Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add the parent directory to the path to import euchre modules
sys.path.append(str(Path.cwd().parent))

from euchre.mass_game_runner import MassGameRunner
from euchre.ai_profiles import AggressiveAI, ConservativeAI, BalancedAI, OpportunisticAI
from euchre.models import PlayerType
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

print("✅ Imports successful")

## Initialize Mass Game Runner

In [ ]:
# Initialize the mass game runner
runner = MassGameRunner()

print("�� Mass Game Runner initialized")

## Configure Game Scenarios

In [ ]:
# Define different AI profile configurations
ai_configs = {
    'aggressive_vs_conservative': {
        'team1': [('North', 'aggressive', 0.8), ('South', 'aggressive', 0.8)],
        'team2': [('East', 'conservative', 0.2), ('West', 'conservative', 0.2)]
    },
    'balanced_vs_opportunistic': {
        'team1': [('North', 'balanced', 0.5), ('South', 'balanced', 0.5)],
        'team2': [('East', 'opportunistic', 0.7), ('West', 'opportunistic', 0.7)]
    },
    'mixed_strategies': {
        'team1': [('North', 'aggressive', 0.8), ('South', 'conservative', 0.2)],
        'team2': [('East', 'balanced', 0.5), ('West', 'opportunistic', 0.7)]
    }
}

print("📋 AI configurations defined:")
for name, config in ai_configs.items():
    print(f"  {name}: {len(config['team1'])} vs {len(config['team2'])} players")

## Run Mass Games

In [ ]:
# Run games for each configuration
games_per_config = 100  # Adjust as needed
total_games = len(ai_configs) * games_per_config

print(f"🎮 Running {total_games} total games...")
print(f"📊 {games_per_config} games per configuration")

all_results = {}

for config_name, config in ai_configs.items():
    print(f"\n🎯 Running {config_name}...")
    
    # Run games for this configuration
    results = runner.run_games(
        num_games=games_per_config,
        team1_config=config['team1'],
        team2_config=config['team2'],
        enable_logging=False,  # Disable logging for mass runs
        progress_bar=True
    )
    
    all_results[config_name] = results
    
    # Print summary
    team1_wins = sum(1 for r in results if r['winner'] == 'team1')
    team2_wins = sum(1 for r in results if r['winner'] == 'team2')
    
    print(f"  📊 Results: Team1: {team1_wins}, Team2: {len(results) - team1_wins}")
    print(f"  🏆 Win Rate: Team1: {team1_wins/len(results):.1%}")

print("\n✅ All games completed!")

## Analyze Results

In [ ]:
# Convert results to DataFrame for analysis
all_games = []
for config_name, results in all_results.items():
    for result in results:
        result['config'] = config_name
        all_games.append(result)

df = pd.DataFrame(all_games)
print(f"📊 Total games analyzed: {len(df)}")
print(f"\n📋 Columns: {list(df.columns)}")
df.head()

## Statistical Analysis

In [ ]:
# Overall statistics
print("�� OVERALL STATISTICS")
print("=" * 50)

# Win rates by configuration
print("\n🏆 Win Rates by Configuration:")
for config_name in ai_configs.keys():
    config_df = df[df['config'] == config_name]
    team1_wins = len(config_df[config_df['winner'] == 'team1'])
    total = len(config_df)
    
    print(f"  {config_name}:")
    print(f"    Team1: {team1_wins}/{total} ({team1_wins/total:.1%})")
    print(f"    Team2: {total - team1_wins}/{total} ({(total - team1_wins)/total:.1%})")

# Game length statistics
if 'game_length' in df.columns:
    print(f"\n⏱️  Game Length Statistics:")
    print(f"  Average: {df['game_length'].mean():.1f} rounds")
    print(f"  Median: {df['game_length'].median():.1f} rounds")
    print(f"  Min: {df['game_length'].min()} rounds")
    print(f"  Max: {df['game_length'].max()} rounds")
    print(f"  Std Dev: {df['game_length'].std():.1f} rounds")

## Visualizations

In [ ]:
# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Win rates by configuration
ax1 = axes[0, 0]
config_names = list(ai_configs.keys())
team1_win_rates = []
team2_win_rates = []

for config_name in config_names:
    config_df = df[df['config'] == config_name]
    team1_wins = len(config_df[config_df['winner'] == 'team1'])
    total = len(config_df)
    
    team1_win_rates.append(team1_wins / total)
    team2_win_rates.append((total - team1_wins) / total)

x = range(len(config_names))
width = 0.35

ax1.bar([i - width/2 for i in x], team1_win_rates, width, label='Team1', alpha=0.8)
ax1.bar([i + width/2 for i in x], team2_win_rates, width, label='Team2', alpha=0.8)
ax1.set_xlabel('Configuration')
ax1.set_ylabel('Win Rate')
ax1.set_title('Win Rates by Configuration')
ax1.set_xticks(x)
ax1.set_xticklabels(config_names, rotation=45)
ax1.set_xticklabels(config_names, rotation=45)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Game length distribution
ax2 = axes[0, 1]
if 'game_length' in df.columns:
    ax2.hist(df['game_length'], bins=20, alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Game Length (Rounds)')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Game Length Distribution')
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Game length data not available', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Game Length Distribution')

# 3. Win rates comparison
ax3 = axes[1, 0]
overall_team1_wins = len(df[df['winner'] == 'team1'])
overall_team2_wins = len(df[df['winner'] == 'team2'])
total_games = len(df)

labels = ['Team1 Wins', 'Team2 Wins']
sizes = [overall_team1_wins, overall_team2_wins]
colors = ['lightblue', 'lightcoral']

ax3.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax3.set_title('Overall Win Distribution')

# 4. Score distributions
ax4 = axes[1, 1]
if 'team1_score' in df.columns and 'team2_score' in df.columns:
    ax4.hist(df['team1_score'], bins=15, alpha=0.7, label='Team1 Scores', density=True)
    ax4.hist(df['team2_score'], bins=15, alpha=0.7, label='Team2 Scores', density=True)
    ax4.set_xlabel('Final Score')
    ax4.set_ylabel('Density')
    ax4.set_title('Score Distributions')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'Score data not available', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Score Distributions')

plt.tight_layout()
plt.show()

## Export Results

In [ ]:
# Export results to CSV
output_file = "mass_game_results.csv"
df.to_csv(output_file, index=False)
print(f"💾 Results exported to {output_file}")

# Export summary statistics
summary_stats = {
    'total_games': len(df),
    'team1_wins': len(df[df['winner'] == 'team1']),
    'team2_wins': len(df[df['winner'] == 'team2']),
}

if 'game_length' in df.columns:
    summary_stats['avg_game_length'] = df['game_length'].mean()
if 'team1_score' in df.columns:
    summary_stats['avg_team1_score'] = df['team1_score'].mean()
if 'team2_score' in df.columns:
    summary_stats['avg_team2_score'] = df['team2_score'].mean()

print("\n📊 Summary Statistics:")
for key, value in summary_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")